# EQUORA NEDİR?

## EQUORA Equity ve Resource Allocation kelimelerinin bileşiminden gelir amacı
## "kaynakların ihtiyaçlara göre adilce dağıtılması"

## Temel mantık marketlerde artacak ürünleri tahmin edip  önce indirip uygulayıp sonra elde ne kadar kalacağını tahmin edip sonrasında bunlara ihtiyacı olacak insanları tahmin edip ikisini eşleştiren bir veri analiz platformudur

# ilk aşama bir markette giren çıkan ürünlerin detayları son kullanım tarihlerinden oluşan bi database'in internetteki verilerin toplanarak ve onlardan esinlenerek oluşturulmasıdır

 # database seçimi olarak postgresql seçildi gerekli tablolar ve viewlar oluşturuldu şimdi internetten çeşitli veriler toplanarak bu tablolar dolduruluyor

kullanılan kaynaklar
- https://www.kaggle.com/datasets/omercolakoglu/turkish-market-sales-dataset-with-9000items/data
- https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K/tree/main/data
- https://www.kaggle.com/datasets/salahuddinahmedshuvo/grocery-inventory-and-sales-dataset/data
- https://www.kaggle.com/datasets/arbaaztamboli/retail-markdown-optimization-discounts-and-sales



# Model yapısının kurulması

## - Kurulacak model yapısının önceliği stok hareketleridir markette olan girdi çıktı hareketleri baz alınır
## - Bu girdi çıktılar kuyruk yapısı olarak düşünülür FEFO yapısı baz alınır ürünlerin skt'sine göre dizilir
## - Kuyruk yapıları feature olarak modellere verilir

1. Ham Hareketten "Gün Sonu Fotoğrafı" ÇıkarmaVeritabanımızda sadece "şu gün 50 girdi, bugün 8 satıldı" gibi fiş kayıtları var. Biz önce her akşam mağaza kapanırken raftaki her bir aktif partinin fotoğrafını çekiyoruz: "Bugün itibarıyla bu partiden kaç adet kaldı, SKT'sine kaç gün var?"

2. Feature'ların Asıl Mantığı (Model Neye Bakıyor?)Bir ürünün elde kalıp kalmayacağını bir insan olarak sen nasıl anlarsan, feature'lar da modele tam olarak aynı 4 soruyu sorduruyor:Önünde kuyruk var mı? ($I^{\text{eff}}_{b,t}$): Aynı sütten rafta iki parti varsa (biri yarın bitiyor, biri 4 gün sonra), müşteri önce eskiyi alır. Yani 4 gün sonraki partinin satılmaya başlaması için önce önündeki eski partinin bitmesi gerekir. Modele sadece kendi stoğunu değil, "önündeki kuyrukla birlikte toplam kaç adet erimesi lazım?" bilgisini veriyoruz.Bu hızla yetişir mi? ($\text{DTE}$, $v$, $\text{DoC}$, $\rho$): Diyelim ki efektif stok $25$ adet, son 7 günde bu sütten günde ortalama $5$ adet satılmış ($v^{(7)}$). Demek ki eldeki stoğun bitmesi $25 / 5 = 5$ gün sürer ($\text{DoC}$). Ama SKT'ye $3$ gün kalmış ($\text{DTE}$). $5 / 3 = 1.67$ ($\rho > 1$). Yani mevcut hızla bu parti kesin elde kalacak. Modelin en çok seveceği sinyal budur.Son günlerde bir terslik var mı? ($m$ — Momentum): Normalde günde $10$ satılan ürün, son 3 günde günde $4$'e düştüyse (ör. yağmur, şok) $m < 1$ olur. Model "satışlar yavaşladı, risk artıyor" diye hemen uyanır.Takvim avantajı var mı? Kalan 3 günün içinde Cumartesi-Pazar varsa (weekend_days_left) satışlar hafta sonu zıplayacağı için parti kurtulabilir; hafta içi durgun günlere denk
geliyorsa kurtulma şansı düşer.


3. Neden İki Ayrı Model (Hurdle) Kullanıyoruz?Çünkü partilerin yaklaşık yarısı hiç fire vermeden (0) bitiyor, diğer yarısında ise mesela 7 veya 15 adet elde kalıyor. Tek model kurarsan ikisinin ortalamasını almaya çalışıp saçmalıyor. Biz işi ikiye bölüyoruz:1. Model (Alarm): "Bu parti elde kalır mı?" ($\hat{p}$ — örn. %78 ihtimalle kalır).2. Model (Miktar): "Kalırsa tahminen kaç adet kalır?" ($\hat{q}$ — örn. 9 adet).Böylece hem riski hem kurtarılacak paket sayısını temizce alıyoruz.



4. Asimetrik Maliyet ve Aksiyon (Ne Zaman Müdahale?)Ürünü çöpe atmak markete %100 zarar yazarken, zaten satılacak ürüne ufak indirim yapmak sadece kârdan azıcık fedakârlık demektir. O yüzden modele "ancak %50 emin olunca alarm ver" demiyoruz; maliyete göre daha hassas bir eşik ($\theta^* \approx 0.30\text{--}0.35$) seçiyoruz. Risk eşiği geçilince de SKT'ye kalan güne göre sistem otomatik yönlendirme yapıyor: 2–4 gün varsa dinamik indirim, 1 gün kaldıysa RL ile sosyal yardım, son günse toplu kurtarma.



5. Hilesiz Test (Sızıntı Koruması)Modeli test ederken geleceği görmesin diye sadece SKT'si geçmiş (kapanmış) partileri alıyoruz, eski tarihlerle eğitip yeni tarihlerle sınıyoruz. Şu an veritabanında duran o aktif 9,395 adetlik stoğu ise hiç eğitime sokmayıp "bugün akşam sistem çalışsa hangi ürünleri yakalardı?" diye canlı demo çıktısı almak için ayırıyoruz.Mantık kafanda tamamen oturdu mu kanka? İstersen bu tasarımın üstüne takıldığın bir nokta varsa konuşalım, yoksa bunu koda dökecek Antigravity promptunu hazırlayayım.